In [ ]:
from HINGE.evaluation import run_synthetic_evaluation

from HINGE.feature_mapping import prepare_expression_features

from sklearn.preprocessing import normalize
import pickle
from pathlib import Path

import matplotlib.pyplot as plt

In [2]:
SAVE_DIR = Path('./hinge_checkpoints')
SAVE_DIR.mkdir(exist_ok=True)

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
n_gene_comps = 128

In [5]:
with open(SAVE_DIR / 'checkpoint_step1.pkl', "rb") as f:
    data_dict=pickle.load(f)

data_dict

{'RNA_panel1': {'s1': AnnData object with n_obs × n_vars = 6989 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's2': AnnData object with n_obs × n_vars = 8978 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's3': AnnData object with n_obs × n_vars = 449221 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's4': AnnData object with n_obs × n_vars = 7442 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'feature_types', 'genome'
      obsm: 'spatial',
  's5': AnnData object with n_obs × n_vars = 16540 × 5101
      obs: 'nearest_superpixel_dist', 'cell_type', 'matched_annotation_dist'
      var: 'gene_ids', 'f

In [18]:
#select which sample to use for the mapping

uni_480_list = data_dict['HE_panel1'][0:1]  # only the first sample for demonstration
uni_5k_list = data_dict['HE_panel2'][0:1]  # only the first sample for demonstration
rna_480_list = [data_dict['RNA_panel1']['s1']]  # only the first sample for demonstration
rna_5k_list = [data_dict['RNA_panel2']['s7']]  # only the first sample for demonstration

In [19]:

uni_480_normalized_list = []

for adata in uni_480_list:
    adata_norm = adata.copy()
    adata_norm.X = normalize(adata.X)
    uni_480_normalized_list.append(adata_norm)

In [20]:
uni_5k_normalized_list = []

for adata in uni_5k_list:
    adata_norm = adata.copy()
    adata_norm.X = normalize(adata.X)
    uni_5k_normalized_list.append(adata_norm)

In [21]:
uni_480_normalized_list

[AnnData object with n_obs × n_vars = 6989 × 2048
     obs: 'nearest_superpixel_dist'
     obsm: 'spatial']

In [22]:
expr_480_list, expr_5k_list, expr_pca, overlap_genes = prepare_expression_features(
    rna_480_list,rna_5k_list, n_comps=n_gene_comps
)


  Preparing expression features (overlapping genes PCA)
  Overlapping genes across all samples: 444
  Fitting expression PCA on (6989, 444) → 128 components...
  Variance retained: 97.6%
Panel 1 expr (6989, 128), Panel 2 expr (6989, 128)


In [ ]:
adata_panel1          = data_dict['RNA_panel1']['s1'],
uni_panel1_normed     = uni_480_normalized_list[0],
adata_panel2          =    data_dict['RNA_panel2']['s7'],
uni_panel2_normed     = uni_5k_normalized_list[0],

# view sample fake data

In [ ]:
from HINGE.evaluation import compute_population_diff_stats, create_synthetic_replicates

stats = compute_population_diff_stats(
        adata_panel1   = adata_panel1,
        uni_panel1_normed     = uni_panel1_normed,
        adata_panel2   = adata_panel2,
        uni_panel2_normed     = uni_panel2_normed,
        overlap_genes  = overlap_genes,
        spatial_key_p1 =  "spatial",
        spatial_key_p2 = "spatial_registered",
    )

    # Step 2: replicates
fakes = create_synthetic_replicates(
        adata_panel1      = adata_panel1,
        uni_panel1_normed = uni_panel1_normed,
        stats             = stats,
        alpha_list        = [0.25,1],
        n_seeds_per_alpha = 1,
        noise_scale       = 1.0,
        spatial_key       =  "spatial",
        RECENTER          = False,
        xy_method="displacement_field",
        xy_k=20,
        xy_local_noise_weight=0.25,
)

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(14, 14), sharex=True, sharey=True)
s=1
# 1. Xenium 480 coordinates
axes[0, 0].scatter(adata_panel1.obsm['spatial'][:, 0], adata_panel1.obsm['spatial'][:, 1], s=s, c="red", alpha=0.4)
axes[0, 0].set_title(f"480 Xenium cell coordinates")

# 2. Xenium 5k transformed coordinates
axes[0, 1].scatter(adata_panel2.obsm['spatial_registered'][:, 0], adata_panel2.obsm['spatial_registered'][:, 1], s=s, c="blue", alpha=0.4)
axes[0, 1].set_title(f"UNI superpixel coordinates (n={len(uni_xy)})")


fake_xy = fakes[0]["xy"]

# 3. fake 
axes[1, 0].scatter(fake_xy[:, 0], fake_xy[:, 1], s=s, c="green", alpha=0.4)
axes[1, 0].set_title(f"Fake xy alpha = 0.25")


fake_xy = fakes[1]["xy"]
# 4. fake 
axes[1, 1].scatter(fake_xy[:, 0], fake_xy[:, 1], s=s, c="green", alpha=0.4)
axes[1, 1].set_title(f"Fake xy alpha = 1")



for ax in axes.ravel():
    ax.set_aspect("equal")
    ax.invert_yaxis()   # image coordinates: origin at top-left; remove if not wanted
    ax.axis("off")

#fig.suptitle(f"Sample: {sample_name}", fontsize=16)
plt.tight_layout()

plt.show()

# run test

In [ ]:

df_results, fakes, stats = run_synthetic_evaluation(
    adata_panel1   = adata_panel1,
    uni_panel1_normed     = uni_panel1_normed,
    adata_panel2   = adata_panel2,
    uni_panel2_normed     = uni_panel2_normed,

    expr_panel1 =expr_480_list[0],
    expr_pca= expr_pca,
    overlap_genes= overlap_genes,
    
    
    SAMPLE_NAME       = "s1",
)

  Computing population-level difference statistics
  Panel1: 6989 cells  |  Panel2: 6881 cells

  Coordinate shift:
    delta_mean_x    = -24.57 um
    delta_mean_y    = -155.12 um
    coord_noise_std = 110.5729 um

  UNI (normalized space, 2048D):
    |delta_mean| median = 0.00258
    delta_std    median = 0.01700

  Expression (444 genes):
    |delta_mean| median = 0.0725
    delta_std    median = 0.1548
  Created s1_t: alpha=0.25  seed=0  mean coord shift=142.84 um  n=6989
  Created s1_t: alpha=0.25  seed=1  mean coord shift=142.42 um  n=6989
  Created s1_t: alpha=0.25  seed=2  mean coord shift=142.57 um  n=6989
  Created s1_t: alpha=0.50  seed=3  mean coord shift=155.66 um  n=6989
  Created s1_t: alpha=0.50  seed=4  mean coord shift=154.38 um  n=6989
  Created s1_t: alpha=0.50  seed=5  mean coord shift=155.37 um  n=6989
  Created s1_t: alpha=0.75  seed=6  mean coord shift=176.51 um  n=6989
  Created s1_t: alpha=0.75  seed=7  mean coord shift=175.74 um  n=6989
  Created s1_t: alpha=